## LoanShield — Part 2: Modeling, Ensemble Comparison & Business Verdict
Ensemble Methods: Decision Tree vs Random Forest vs XGBoost

Dataset : HMEQ (Home Equity Loan Default) — 5,960 rows

Domain  : FinTech / Lending Risk

This notebook picks up after EDA/cleaning/preprocessing (see **01_eda_fraud_landscape.ipynb**). It loads the processed artifacts, trains three models, compares them, derives a cost-optimised decision threshold, and summarises the business verdict.


##CELL 1 - Import Requried  Library

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection      import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing        import LabelEncoder, StandardScaler
from sklearn.tree                 import DecisionTreeClassifier
from sklearn.ensemble             import RandomForestClassifier
from xgboost                      import XGBClassifier
from sklearn.metrics              import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)


In [ ]:
#  Dark-theme plot settings (consistent with portfolio)
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2a4a",
    "figure.dpi":       150,
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#444",
})

In [ ]:
#  Colour palette
COLOR_FRAUD   = "#ef5350"   # red   — fraudulent / high-risk
COLOR_LEGIT   = "#66bb6a"   # green — legitimate / low-risk
COLOR_DT      = "#78909c"   # gray  — baseline Decision Tree
COLOR_RF      = "#42a5f5"   # blue  — Random Forest
COLOR_XGB     = "#ffca28"   # amber — XGBoost
COLOR_NEUTRAL = "#7e57c2"   # purple — neutral highlights


In [ ]:
# Load processed artifacts from the EDA notebook
import pickle

import os
def find_file(filename):
    if os.path.exists(filename):
        return filename
    # Search in subdirectories
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    # Search in parent directories up to 3 levels
    path = '..'
    for _ in range(3):
        target = os.path.abspath(os.path.join(path, filename))
        if os.path.exists(target):
            return target
        path = os.path.join(path, '..')
    return filename
with open(find_file("loanshield_artifacts.pkl"), "rb") as f:
    artifacts = pickle.load(f)

df_clean     = artifacts["df_clean"]
X_train      = artifacts["X_train"]
X_test       = artifacts["X_test"]
y_train      = artifacts["y_train"]
y_test       = artifacts["y_test"]
y            = artifacts["y"]
FEATURE_COLS = artifacts["FEATURE_COLS"]

print("✓ Loaded processed artifacts from loanshield_artifacts.pkl")
print(f"  X_train: {X_train.shape}   X_test: {X_test.shape}")


# CELL 7 — Model 1 : Decision Tree(Baseline)





In [ ]:
print("  Model 1 of 3: Decision Tree ")

dt = DecisionTreeClassifier(
    max_depth    = 6,
    class_weight = "balanced",
    random_state = 42
)
dt.fit(X_train, y_train)

In [ ]:
y_pred_dt  = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]
auc_dt     = roc_auc_score(y_test, y_proba_dt)
ap_dt      = average_precision_score(y_test, y_proba_dt)

print(f"\n  ROC-AUC          : {auc_dt:.4f}")
print(f"  Average Precision: {ap_dt:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=["Legit", "Fraud/Default"]))

In [ ]:
# Cross-validation to expose variance (tree overfitting problem)

cv_scores_dt = cross_val_score(dt, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_dt.mean():.4f} ± {cv_scores_dt.std():.4f}")
print(f"  (High std = high variance — this is the problem ensembles fix)")

# CELL 8 — Model 2 :  Random Forest (Bagging)


In [ ]:
print("  Model 2 of 3: Random Forest (Bagging)")

rf = RandomForestClassifier(
    n_estimators = 200,
    max_features = "sqrt",
    max_depth    = 12,
    class_weight = "balanced",
    n_jobs       = -1,
    random_state = 42
)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
auc_rf     = roc_auc_score(y_test, y_proba_rf)
ap_rf      = average_precision_score(y_test, y_proba_rf)

print(f"\n  ROC-AUC          : {auc_rf:.4f}")
print(f"  Average Precision: {ap_rf:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Legit", "Fraud/Default"]))

In [ ]:
cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")
print(f"  (Lower std than DT = lower variance — bagging working as intended)")

# CELL 9 — Model 3 : XGBoost (Gradient Boosting)


In [ ]:
print("  Model 3 of 3: XGBoost (Gradient Boosting)")

scale_pos_wt = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\n  scale_pos_weight = {scale_pos_wt:.2f}  (fraud upweighted by {scale_pos_wt:.1f}x)")

xgb = XGBClassifier(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 5,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = scale_pos_wt,
    eval_metric       = "auc",
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1
)

xgb.fit(X_train, y_train, verbose=False)

In [ ]:
y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
auc_xgb     = roc_auc_score(y_test, y_proba_xgb)
ap_xgb      = average_precision_score(y_test, y_proba_xgb)

print(f"\n  ROC-AUC          : {auc_xgb:.4f}")
print(f"  Average Precision: {ap_xgb:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=["Legit", "Fraud/Default"]))

In [ ]:
cv_scores_xgb = cross_val_score(xgb, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")

# Cell 10 —  Model Comparison Dashboard


In [ ]:
#ROC Curves rendered

fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, color, auc_val in [
    ("Decision Tree",   y_proba_dt,  COLOR_DT,  auc_dt),
    ("Random Forest",   y_proba_rf,  COLOR_RF,  auc_rf),
    ("XGBoost",         y_proba_xgb, COLOR_XGB, auc_xgb),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name}  AUC={auc_val:.4f}")
ax.plot([0, 1], [0, 1], color="#555", lw=1, ls="--", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Three Models", color="#e0e0e0")
ax.legend(fontsize=9, framealpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()
print("✓ Panel 1 — ROC Curves rendered")

In [ ]:
# Precision-Recall Curves rendered

fig, ax = plt.subplots(figsize=(9, 7))
for name, proba, color, ap_val in [
    ("Decision Tree",   y_proba_dt,  COLOR_DT,  ap_dt),
    ("Random Forest",   y_proba_rf,  COLOR_RF,  ap_rf),
    ("XGBoost",         y_proba_xgb, COLOR_XGB, ap_xgb),
]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, color=color, lw=2, label=f"{name}  AP={ap_val:.4f}")
baseline_pr = y_test.mean()
ax.axhline(baseline_pr, color="#555", lw=1, ls="--",
           label=f"Random baseline (AP={baseline_pr:.2f})")
ax.set_xlabel("Recall (Fraud Caught %)")
ax.set_ylabel("Precision (Of Flagged, % Are Real Fraud)")
ax.set_title("Precision-Recall Curves — Focus on Fraud Class", color="#e0e0e0")
ax.legend(fontsize=9, framealpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()
print("✓ Panel 2 — Precision-Recall Curves rendered")

In [ ]:
# Confusion Matrix rendered

fig, ax = plt.subplots(figsize=(7, 5))
best_name = max(
    [("Decision Tree", auc_dt, y_pred_dt),
     ("Random Forest", auc_rf, y_pred_rf),
     ("XGBoost",       auc_xgb, y_pred_xgb)],
    key=lambda x: x[1]
)
cm = confusion_matrix(y_test, best_name[2])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"],
            ax=ax, linewidths=0.5, linecolor="#333",
            annot_kws={"size": 14, "weight": "bold"})
ax.set_title(f"Confusion Matrix — {best_name[0]} (Best Model)", color="#e0e0e0")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
tn, fp, fn, tp = cm.ravel()
ax.text(0.02, 0.02,
        f"TN={tn}  FP={fp}\nFN={fn}  TP={tp}",
        transform=ax.transAxes, fontsize=9, color="#e0e0e0",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#2a2a4a", edgecolor="#555"))
plt.tight_layout()
plt.show()
print("✓ Panel 3 — Confusion Matrix rendered")

# CELL 11 — BUSINESS VERDICT

In [ ]:
print("BUSINESS VERDICT")
print("Cost-Optimised Threshold Analysis")

avg_loan_amount = df_clean["LOAN"].median()

print(f"\nAvg loan amount  : ${avg_loan_amount:,.0f}")
print(f"FN cost (default): ${avg_loan_amount:,.0f}  (lose full principal)")

In [ ]:
annual_interest = avg_loan_amount * 0.065

print(f"  Annual interest  : ${annual_interest:,.0f}")
print(f"  FP cost (missed) : ${annual_interest:,.0f}   (lose 1-yr interest)\n")
print(f"  Cost ratio FN:FP : {avg_loan_amount/annual_interest:.1f}X")

In [ ]:
proba_best  = y_proba_xgb
thresholds  = np.arange(0.05, 0.95, 0.01)
total_costs = []
precisions  = []
recalls     = []
f1_scores   = []

In [ ]:
for t in thresholds:
    y_pred_t             = (proba_best >= t).astype(int)
    cm_t                 = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    cost = (fn_t * avg_loan_amount) + (fp_t * annual_interest)
    total_costs.append(cost)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))

In [ ]:
opt_idx       = np.argmin(total_costs)
opt_threshold = thresholds[opt_idx]
opt_cost      = total_costs[opt_idx]
default_idx   = np.argmin(np.abs(thresholds - 0.5))
default_cost  = total_costs[default_idx]
savings       = default_cost - opt_cost



In [ ]:
print(f"  Default threshold (0.50) cost : ${default_cost:,.0f}")
print(f"  Optimal threshold ({opt_threshold:.2f}) cost  : ${opt_cost:,.0f}")
print(f"  Estimated savings             : ${savings:,.0f} per test period")
print(f"\n  At optimal threshold {opt_threshold:.2f}:")

In [ ]:
y_pred_opt = (proba_best >= opt_threshold).astype(int)
print(classification_report(y_test, y_pred_opt, target_names=["Legit", "Fraud/Default"]))

# CELL 12 — Feature Importance Comparison (RF vs XGBoost)

In [ ]:
print("Feature Importance Analysis")

rf_importance  = pd.Series(rf.feature_importances_,  index=FEATURE_COLS).sort_values(ascending=True)

print("\n  Top 5 features — Random Forest:")
print(rf_importance.tail(5).round(4).to_string())


In [ ]:
xgb_importance = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

print("Top 5 features — XGBoost:\n")
print(xgb_importance.tail(5).round(4).to_string())

# CELL 13 : Full Business Verdict Dashboard


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(thresholds, [c / 1000 for c in total_costs], color=COLOR_XGB, lw=2)
ax.axvline(opt_threshold, color=COLOR_FRAUD, lw=2, ls="--",
           label=f"Optimal: {opt_threshold:.2f}")
ax.axvline(0.5, color="#aaa", lw=1.5, ls=":",
           label="Default: 0.50")
ax.scatter([opt_threshold], [opt_cost / 1000], color=COLOR_FRAUD, s=120, zorder=5)
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Total Business Cost ($K)")
ax.set_title("Threshold Optimisation — Minimise Loan Losses", color="#e0e0e0")
ax.legend(fontsize=9, framealpha=0.4)
ax.text(opt_threshold + 0.02, (opt_cost / 1000) * 1.05,
        f"Saves ${savings:,.0f}\nvs default",
        color=COLOR_LEGIT, fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()
print("✓ Panel 1 — Threshold Optimisation rendered")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(thresholds, recalls,    color=COLOR_FRAUD, lw=2, label="Recall (fraud caught)")
ax.plot(thresholds, precisions, color=COLOR_RF,    lw=2, label="Precision (flag accuracy)")
ax.plot(thresholds, f1_scores,  color=COLOR_XGB,   lw=2, label="F1 Score", ls="--")
ax.axvline(opt_threshold, color="#e0e0e0", lw=1.5, ls=":",
           label=f"Optimal: {opt_threshold:.2f}")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision vs Recall Trade-off by Threshold", color="#e0e0e0")
ax.legend(fontsize=9, framealpha=0.4)
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()
print("✓ Panel 2 — Precision vs Recall Trade-off rendered")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
feat_df = pd.DataFrame({
    "Random Forest": rf_importance,
    "XGBoost":       xgb_importance
}).sort_values("Random Forest", ascending=True)

y_pos       = np.arange(len(feat_df))
bar_height  = 0.35
ax.barh(y_pos - bar_height/2, feat_df["Random Forest"], height=bar_height,
        color=COLOR_RF, alpha=0.85, label="Random Forest (MDI)")
ax.barh(y_pos + bar_height/2, feat_df["XGBoost"], height=bar_height,
        color=COLOR_XGB, alpha=0.85, label="XGBoost (Gain)")
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_df.index, fontsize=9)
ax.set_xlabel("Feature Importance Score")
ax.set_title("Feature Importance — RF vs XGBoost\n(Where both agree = robust signal)",
             color="#e0e0e0")
ax.legend(fontsize=9, framealpha=0.4)
plt.tight_layout()
plt.show()
print("✓ Panel 3 — Feature Importance Comparison rendered")

# CELL 14 — FINAL SUMMARY


In [ ]:
print("  ✓ LOANSHIELD PIPELINE COMPLETE\n")

winner = max(
    [("Decision Tree", auc_dt), ("Random Forest", auc_rf), ("XGBoost", auc_xgb)],
    key=lambda x: x[1]
)

lines = [
    "",
    "  PROJECT   : LoanShield - Lending Fraud & Default Detection Engine",
    f"  DATASET   : HMEQ (Home Equity Loan Default) - {len(df_clean):,} rows",
    f"  TARGET    : BAD (1=default/fraud, 0=legitimate) | Fraud rate: {y.mean()*100:.1f}%",
    "",
    "  MODELS TRAINED:",
    f"    1. Decision Tree  (Baseline)  - AUC: {auc_dt:.4f}",
    f"    2. Random Forest  (Bagging)   - AUC: {auc_rf:.4f}  +{auc_rf-auc_dt:.4f} vs baseline",
    f"    3. XGBoost        (Boosting)  - AUC: {auc_xgb:.4f}  +{auc_xgb-auc_dt:.4f} vs baseline",
    "",
    f"  WINNER    : {winner[0]}  (ROC-AUC: {winner[1]:.4f})",
    "",
    "  BUSINESS IMPACT:",
    f"    Default threshold  (0.50): Total cost estimate - ${default_cost:,.0f}",
    f"    Optimal threshold  ({opt_threshold:.2f}): Total cost estimate - ${opt_cost:,.0f}",
    f"    Estimated savings         : ${savings:,.0f} per lending cycle",
    "",
    "  TOP FRAUD SIGNALS (both models agree):",
    "    1. DEBTINC  - Debt-to-income ratio     (highest weight in both)",
    "    2. DEROG    - Derogatory credit reports",
    "    3. DELINQ   - Past delinquencies",
    "    4. CLAGE    - Age of credit history",
    "    5. VALUE    - Property value (collateral)",
    "",
    "  CONCEPTS DEMONSTRATED:",
    "    ✓ Class imbalance handling (class_weight + scale_pos_weight)",
    "    ✓ Decision Tree - high variance baseline",
    "    ✓ Random Forest - bagging, bootstrap sampling, feature randomness",
    "    ✓ XGBoost       - boosting, sequential error correction, regularisation",
    "    ✓ Stratified K-Fold cross-validation",
    "    ✓ ROC-AUC vs Precision-Recall AUC (imbalanced class evaluation)",
    "    ✓ Business cost matrix - FN vs FP financial impact",
    "    ✓ Threshold optimisation - minimise total loan losses",
    "    ✓ Feature importance comparison across two ensemble families",
]
print("\n".join(lines))